In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re

In [2]:
# df1 = pd.read_csv(r"..\data\processed\1335586090760798259.csv")
# df2 = pd.read_csv(r"..\data\processed\private.csv")
# df = pd.concat([df1, df2], ignore_index=True)
df = pd.read_csv(r"..\data\processed\private.csv")
df['Content'] = df['Content'].astype("str")
df.head()

,Author,Content
0,fa09cb53,new year bitch
1,fa09cb53,eat piss
2,fa09cb53,and cum
3,fa09cb53,wholesome 2021 moment
4,e1688f3c,the year is 2021


In [3]:
pattern = r"<[^>]+>"

filtered_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# remove attachments, links, non-standard emojis
filtered_df = filtered_df[~filtered_df["Content"].str.contains(pattern, na=False)]



# Keep only authors with >= 100 messages
author_counts = filtered_df['Author'].value_counts()
filtered_df = filtered_df[filtered_df['Author'].isin(author_counts[author_counts >= 100].index)]

# formatting
filtered_df["Content"] = (
    filtered_df["Content"]
    # **bold** → bold
    .str.replace(r"\*\*(.*?)\*\*", r"\1", regex=True)
    # *italic* → italic
    .str.replace(r"\*(.*?)\*", r"\1", regex=True)
    # ||spoiler|| → spoiler
    .str.replace(r"\|\|(.*?)\|\|", r"\1", regex=True)
)
filtered_df = filtered_df[
    ~filtered_df["Content"].str.contains(pattern, na=False)
]

# length filter
filtered_df = filtered_df[filtered_df["Content"].str.len() >= 20]
filtered_df = filtered_df[filtered_df["Content"].str.len() <= 100]

filtered_df = filtered_df.dropna(subset=['Content'])
filtered_df['Content'] = filtered_df['Content'].astype("str")

SAMPLE_SIZE = 20000
if len(filtered_df) > SAMPLE_SIZE:
    filtered_df_sample = filtered_df.sample(n=SAMPLE_SIZE, random_state=42)

In [8]:
train_df, val_df = train_test_split(filtered_df_sample, test_size=0.2, random_state=42)
train_df.to_csv(r'..\data\train\retriever_train_undersampled.csv', index=False)
val_df.to_csv(r'..\data\train\retriever_val_undersampled.csv', index=False)

train_df, val_df = train_test_split(filtered_df, test_size=0.2, random_state=42)
train_df.to_csv(r'..\data\train\retriever_train.csv', index=False)
val_df.to_csv(r'..\data\train\retriever_val.csv', index=False)

In [4]:
filtered_df.to_csv(r'..\data\rag\private_embeddings_full.csv', index=False)